<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# FXCM Algorithmic Trading Initiative

## RESTful API & Automated Trading

**Dr. Yves J. Hilpisch**


The Python Quants GmbH

<img src="http://hilpisch.com/images/finaince_visual_low.png" width=300px align=left>

## Risk Disclaimer

<font size="-1">
Trading forex/CFDs on margin carries a high level of risk and may not be suitable for all investors as you could sustain losses in excess of deposits. Leverage can work against you. Due to the certain restrictions imposed by the local law and regulation, German resident retail client(s) could sustain a total loss of deposited funds but are not subject to subsequent payment obligations beyond the deposited funds. Be aware and fully understand all risks associated with the market and trading. Prior to trading any products, carefully consider your financial situation and experience level. Any opinions, news, research, analyses, prices, or other information is provided as general market commentary, and does not constitute investment advice. FXCM & TPQ will not accept liability for any loss or damage, including without limitation to, any loss of profit, which may arise directly or indirectly from use of or reliance on such information.
</font>

## Reading FXCM Tick Data

To install `fxcmpy`:

    pip install fxcmpy

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_addon.git
import sys
sys.path.append('python_for_algo_trading_addon')


In [ ]:
# !pip install fxcmpy

In [ ]:
import fxcmpy

In [ ]:
fxcmpy.__version__

### The Tick Reader Class

In [ ]:
from fxcmpy import fxcmpy_tick_data_reader as fxcm_tick_reader

### Available Symbols

In [ ]:
fxcm_tick_reader.get_available_symbols()

### Retrieving Tick Data

In [ ]:
import pandas as pd
import datetime as dt

In [ ]:
start = dt.datetime(2017, 12, 5)
stop = dt.datetime(2017, 12, 6)

In [ ]:
%time td = fxcm_tick_reader('EURUSD', start, stop)

In [ ]:
type(td)

In [ ]:
td.get_raw_data().info()

In [ ]:
td.get_raw_data().tail(10)

## Working with the Tick Data

In [ ]:
%time td.get_data().info()

In [ ]:
%time td.get_data().info()

In [ ]:
%%time
data = td.get_data(start='2017-12-07 08:00:00', end='2017-12-08 16:00:00')
data.info()

In [ ]:
from pylab import plt
plt.style.use('seaborn-v0_8')
%matplotlib inline

In [ ]:
data['Bid'].plot(figsize=(10, 6), lw=0.8);

## Connecting to the FXCM RESTful API

To register for a **demo account** go to:

https://tradingstation.fxcm.com
    
The documenation of `fxcmpy` is found under:

http://fxcmpy.tpq.io

In [ ]:
con = fxcmpy.fxcmpy(config_file='fxcm.cfg')

In [ ]:
instruments = con.get_instruments()
print(instruments)

## Historical Data

### Retrieving Historical Data

In [ ]:
candles = con.get_candles('USD/JPY', period='D1', number=10)

In [ ]:
# 10 most recent days | daily
candles

The parameter `period` must be one of `m1, m5, m15, m30, H1, H2, H3, H4, H6, H8, D1, W1` or `M1`.

In [ ]:
# 50 most recent 1 minute bars
candles = con.get_candles('EUR/USD', period='m1', number=50)
candles.tail(10)

### Visualization of the Data

In [ ]:
data = candles[['askopen', 'askhigh', 'asklow', 'askclose']]
data.columns = ['open', 'high', 'low', 'close']
data.info()

In [ ]:
import cufflinks as cf
cf.set_config_file(offline=True)

In [ ]:
qf = cf.QuantFig(data, title='EUR/USD', legend='top',
                 name='EUR/USD')

In [ ]:
qf.iplot()

In [ ]:
qf.add_bollinger_bands(periods=10, boll_std=2, colors=['magenta', 'grey'], fill=True)
qf.data.update()

In [ ]:
qf.iplot()

## Using Machine Learning for Market Prediction

<b style="color: red;">The following example is simplified and for illustration purposes only. Among others, it does not consider transactions costs or bid-ask spreads.</b>

In [ ]:
import datetime
import numpy as np
import pandas as pd

### Data Retrieval

In [ ]:
# 1,000 most recent 1 minute bars
candles = con.get_candles('EUR/USD', period='m1', number=1000)

In [ ]:
data = pd.DataFrame(candles[['askclose', 'bidclose']].mean(axis=1),
                    columns=['midclose'])

In [ ]:
data.tail()

In [ ]:
data.iplot()

### Feature Preparation

In [ ]:
data['returns'] = np.log(data / data.shift(1))

In [ ]:
lags = 8
cols = []
for lag in range(1, lags + 1):
    col = 'lag_%s' % lag
    data[col] = data['returns'].shift(lag)
    cols.append(col)

In [ ]:
from pylab import plt
plt.style.use('seaborn-v0_8')
%matplotlib inline

In [ ]:
data['direction'] = np.sign(data['returns'])
to_plot = ['midclose', 'returns', 'direction']
data[to_plot].iloc[:100].plot(figsize=(10, 6),
        subplots=True, style=['-', '-', 'ro'], title='EUR/USD');

In [ ]:
# the "patterns" = 2 ** lags
np.digitize(data[cols].dropna(), bins=[0])[:10]

In [ ]:
2 ** 8

In [ ]:
data.dropna(inplace=True)

### Support Vector Machines

In [ ]:
from sklearn import svm

In [ ]:
model = svm.SVC(C=100, probability=True, gamma='auto')

In [ ]:
data.info()

In [ ]:
%time model.fit(np.sign(data[cols]), np.sign(data['returns']))

#### Predicting Market Direction 

In [ ]:
pred = model.predict(np.sign(data[cols]))
pred[:15]

#### Probabilities for Market Direction

In [ ]:
pred_proba = model.predict_proba(np.sign(data[cols]))

In [ ]:
model.classes_

In [ ]:
pred_proba[:10]

In [ ]:
probabilities = pd.DataFrame(pred_proba, columns=list(model.classes_))

In [ ]:
probabilities.hist(cumulative=False, bins=30, figsize=(8, 8));

### Vectorized Backtesting

In [ ]:
data['position'] = pred

In [ ]:
data['strategy'] = data['position'] * data['returns']

In [ ]:
# unleveraged | no bid-ask spread or transaction costs | only in-sample
data[['returns', 'strategy']].cumsum().apply(np.exp).plot(figsize=(10, 6));

In [ ]:
data['position'].value_counts()

## Train Test Split

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

### Split Feature Sets

In [ ]:
mu = data['returns'].mean()
v = data['returns'].std()
bins = [mu - v, mu, mu + v]
train_x, test_x, train_y, test_y = train_test_split(
    data[cols].apply(lambda x: np.digitize(x, bins=bins)),                   
    np.sign(data['returns']),
    test_size=0.50, random_state=111)

In [ ]:
train_x.sort_index(inplace=True)
train_y.sort_index(inplace=True)
test_x.sort_index(inplace=True)
test_y.sort_index(inplace=True)

In [ ]:
# the patterns = buckets ** lags
train_x.head(10)

In [ ]:
test_x.head(5)

In [ ]:
4 ** 8

In [ ]:
ax = data['midclose'][train_x.index].plot(style=['b.'], figsize=(10, 6))
data['midclose'][test_x.index].plot(style=['r.'], ax=ax)
data['midclose'].plot(ax=ax, lw=0.5, style=['k--']);

In [ ]:
ax = data['midclose'].iloc[-100:][train_x.index].plot(style=['bo'], figsize=(10, 6))
data['midclose'].iloc[-100:][test_x.index].plot(style=['ro'], ax=ax)
data['midclose'].iloc[-100:].plot(ax=ax, lw=0.5, style=['k--']);

### Model Fitting & Prediction

In [ ]:
model.fit(train_x, train_y)

In [ ]:
train_pred = model.predict(train_x)

In [ ]:
accuracy_score(train_y, train_pred)

In [ ]:
test_pred = model.predict(test_x)

In [ ]:
accuracy_score(test_y, test_pred)

### Vectorized Backtesting &mdash; Direct Predictions

In [ ]:
pred = model.predict(np.digitize(data[cols], bins=bins))
pred[:15]

In [ ]:
data['position'] = pred

In [ ]:
data['strategy'] = data['position'] * data['returns']

In [ ]:
# in-sample
data.loc[train_x.index][['returns', 'strategy']].cumsum().apply(np.exp).plot(figsize=(10, 6));

In [ ]:
# out-of-sample
data.loc[test_x.index][['returns', 'strategy']].cumsum().apply(np.exp).plot(figsize=(10, 6));

## Retrieving Streaming Data

### Market Data Stream

In [ ]:
def output(data, dataframe):
    print('%3d | %s | %s, %s, %s, %s, %s' 
          % (len(dataframe), data['Symbol'],
             pd.to_datetime(int(data['Updated']), unit='ms'), 
             data['Rates'][0], data['Rates'][1],
             data['Rates'][2], data['Rates'][3]))

In [ ]:
con.subscribe_market_data('EUR/USD', (output,))

In [ ]:
con.get_last_price('EUR/USD')

In [ ]:
con.unsubscribe_market_data('EUR/USD')

## Placing Orders via the RESTful API

In [ ]:
con.get_open_positions()

In [ ]:
order = con.create_market_buy_order('EUR/USD', 5)

In [ ]:
order.get_currency()

In [ ]:
order.get_isBuy()

In [ ]:
cols = ['tradeId', 'amountK', 'currency', 'grossPL', 'isBuy']

In [ ]:
con.get_open_positions()[cols]

In [ ]:
order = con.create_market_buy_order('USD/JPY', 5)

In [ ]:
con.get_open_positions()[cols]

In [ ]:
order = con.create_market_sell_order('EUR/USD', 2.5)

In [ ]:
order = con.create_market_buy_order('USD/JPY', 5)

In [ ]:
con.get_open_positions()[cols]

In [ ]:
order = con.create_market_sell_order('EUR/USD', 5)

In [ ]:
con.get_open_positions()[cols]

In [ ]:
con.close_all_for_symbol('USD/JPY')

In [ ]:
con.get_open_positions()[cols]

In [ ]:
con.close_all()

In [ ]:
con.get_open_positions()

## Automated Trading

### Model Fitting 

In [ ]:
lags = 3

In [ ]:
def generate_features(df, lags):
    df['Returns'] = np.log(df['Mid'] / df['Mid'].shift(1))
    cols = []
    for lag in range(1, lags + 1):
        col = 'lag_%s' % lag
        df[col] = np.sign(df['Returns'].shift(lag))
        cols.append(col)
    df.dropna(inplace=True)
    return df, cols

In [ ]:
# 1,000 most recent 1 minute bars
candles = con.get_candles('EUR/USD', period='m1', number=1000)

In [ ]:
data = pd.DataFrame(candles[['askclose', 'bidclose']].mean(axis=1),
                    columns=['Mid'])

In [ ]:
data, cols = generate_features(data, lags)

In [ ]:
data.tail()

In [ ]:
model.fit(data[cols], np.sign(data['Returns']))

In [ ]:
model.predict(data[cols])[:10]

### The Basic Idea

In [ ]:
data[cols].iloc[-1].values

In [ ]:
model.predict(data[cols].iloc[-1].values.reshape(1, -1))

### Automated Implementation

In [ ]:
con2 = fxcmpy.fxcmpy(config_file='fxcm.cfg')

In [ ]:
to_show = ['tradeId', 'amountK', 'currency', 'grossPL', 'isBuy']

In [ ]:
ticks = 0
position = 0
tick_data = pd.DataFrame()
tick_resam = pd.DataFrame()

In [ ]:
def automated_trading(data, df):
    global lags, model, ticks
    global tick_data, tick_resam, to_show
    global position
    ticks += 1
    t = datetime.datetime.now()
    if ticks % 5 == 0:
        print('%3d | %s | %7.5f | %7.5f' % (ticks, str(t.time()),
                                            data['Rates'][0], data['Rates'][1]))

    # COLLECTING TICK DATA
    tick_data = tick_data.append(pd.DataFrame(
            {'Bid': data['Rates'][0], 'Ask': data['Rates'][1],
             'High': data['Rates'][2], 'Low': data['Rates'][3]},
            index=[t]))

    # RESAMPLING TICK DATA
    tick_resam = tick_data[['Bid', 'Ask']].resample('5s', label='right').last().ffill()
    tick_resam['Mid'] = tick_resam.mean(axis=1)
    if len(tick_resam) > lags + 2:
        # GENERATING A SIGNAL
        tick_resam, cols = generate_features(tick_resam, lags)
        tick_resam['Prediction'] = model.predict(tick_resam[cols])
        # ENTERING A LONG POSITION
        if tick_resam['Prediction'].iloc[-2] >= 0 and position == 0:
            print('going long (for the first time)')
            position = 1
            order = con2.create_market_buy_order('EUR/USD', 2.5)
        elif tick_resam['Prediction'].iloc[-2] >= 0 and position == -1:
            print('going long')
            position = 1
            order = con2.create_market_buy_order('EUR/USD', 5)
        # ENTERING A SHORT POSITION 
        elif tick_resam['Prediction'].iloc[-2] <= 0 and position == 0:
            print('going short (for the first time)')
            position = -1
            order = con2.create_market_sell_order('EUR/USD', 2.5)
        elif tick_resam['Prediction'].iloc[-2] <= 0 and position == 1:
            print('going short')
            position = -1
            order = con2.create_market_sell_order('EUR/USD', 50)
    if ticks > 100:
        con.unsubscribe_market_data('EUR/USD')
        print('closing out all positions')
        try:
            con2.close_all()
        except:
            pass

In [ ]:
con.subscribe_market_data('EUR/USD', (automated_trading,))

In [ ]:
tick_data.tail()

In [ ]:
tick_resam.tail()

In [ ]:
try:
    print(con2.get_open_positions()[to_show])
except:
    print('no open positions')

In [ ]:
# con2.get_open_positions()

In [ ]:
# con2.get_closed_positions()[to_show]

In [ ]:
# con.unsubscribe_market_data('EUR/USD')

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>